# 01 - Data Quality

**Objective:** Confirm the raw Financial Budgeting dataset is complete, consistent, and fit for analysis before any KPI work begins.

**Business context:** This dataset tracks department-level budget allocation, utilization, and revenue forecasting. Before trusting any downstream KPI, we need to know the data has no missing values, no duplicate records, and valid categorical domains.

In [1]:
import sys
sys.path.insert(0, '../src')
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from data_loading import load_dataset
from data_cleaning import clean_dataset
from feature_engineering import engineer_features


## Load raw data

In [2]:
raw = load_dataset()
print(f'{len(raw):,} rows, {len(raw.columns)} columns')
raw.head()

6,780 rows, 20 columns


,Record_ID,Fiscal_Quarter,Department,Expense_Category,Budget_Allocated,Budget_Utilized,Monthly_Expense,Revenue_Forecast,Actual_Revenue,Budget_Variance,Revenue_Variance,Expense_Growth_Rate,Seasonal_Index,Spending_Volatility,Rolling_Expense_Mean,Rolling_Revenue_Mean,Inflation_Rate,Market_Index,Allocation_Efficiency,Budget_Status
0,1,Q3,Sales,Infrastructure,55526.37,76094.72,13066.48,298866.84,110913.24,-14346.85,-22619.87,7.20,1.07,0.061,7334.48,112248.32,2.81,123.98,79.81,Efficient
1,2,Q4,Finance,Operations,216798.15,143684.41,15238.29,102690.13,190508.00,13299.75,-21500.67,-0.41,1.25,0.277,6078.59,210648.54,4.70,118.97,84.49,Inefficient
2,3,Q1,R&D,Technology,78057.00,61616.75,14073.51,268312.08,143644.82,7240.65,-17170.75,0.46,0.98,0.409,9105.74,144525.26,4.75,120.69,70.44,Moderate
3,4,Q3,Sales,Operations,227312.82,54071.11,5454.20,184356.69,209880.92,-4564.39,-17079.68,7.24,1.05,0.266,15587.12,172581.80,4.36,98.00,96.90,Inefficient
4,5,Q3,Marketing,Training,195954.53,146367.90,13113.98,75224.25,159960.48,-5646.62,-12602.42,-4.42,1.12,0.076,5282.20,129766.17,5.67,107.21,95.84,Inefficient


## Structural checks

In [3]:
print('Duplicated rows:', raw.duplicated().sum())
print('Duplicate Record_ID:', raw['Record_ID'].duplicated().sum())
print()
print('Missing values per column:')
print(raw.isna().sum())

Duplicated rows: 0
Duplicate Record_ID: 0

Missing values per column:
Record_ID                0
Fiscal_Quarter           0
Department               0
Expense_Category         0
Budget_Allocated         0
Budget_Utilized          0
Monthly_Expense          0
Revenue_Forecast         0
Actual_Revenue           0
Budget_Variance          0
Revenue_Variance         0
Expense_Growth_Rate      0
Seasonal_Index           0
Spending_Volatility      0
Rolling_Expense_Mean     0
Rolling_Revenue_Mean     0
Inflation_Rate           0
Market_Index             0
Allocation_Efficiency    0
Budget_Status            0
dtype: int64


## Categorical domains

In [4]:
for c in ['Fiscal_Quarter','Department','Expense_Category','Budget_Status']:
    print(c, '->', sorted(raw[c].unique().tolist()))

Fiscal_Quarter -> ['Q1', 'Q2', 'Q3', 'Q4']
Department -> ['Finance', 'HR', 'IT', 'Logistics', 'Marketing', 'Operations', 'R&D', 'Sales']
Expense_Category -> ['Infrastructure', 'Maintenance', 'Operations', 'Salaries', 'Technology', 'Training']
Budget_Status -> ['Efficient', 'Inefficient', 'Moderate']


## Numeric ranges

In [5]:
raw.describe().T

,count,mean,std,min,25%,50%,75%,max
Record_ID,6780.0,3390.500000,1957.361745,1.00,1695.7500,3390.500,5085.2500,6780.00
Budget_Allocated,6780.0,150410.476702,57946.250881,50031.55,100220.7950,150604.835,201295.8475,249984.97
Budget_Utilized,6780.0,144378.132059,56295.440298,45009.38,95848.7500,144796.205,192691.1550,239980.69
Monthly_Expense,6780.0,11405.499350,4876.714490,3001.15,7189.9150,11417.370,15583.4125,19995.45
Revenue_Forecast,6780.0,184456.468705,66359.722363,70001.27,126532.7225,184954.245,241400.3500,299951.56
Actual_Revenue,6780.0,177290.342900,64987.014994,65003.77,121223.2725,176029.780,234390.9025,289993.73
Budget_Variance,6780.0,-53.121532,11577.639085,-19995.81,-10164.6950,78.030,9957.1825,19998.28
Revenue_Variance,6780.0,105.506693,14386.610587,-24999.58,-12134.3100,259.030,12607.7425,24996.98
Expense_Growth_Rate,6780.0,2.494870,4.328066,-5.00,-1.2000,2.445,6.2900,10.00
Seasonal_Index,6780.0,1.050280,0.142461,0.80,0.9300,1.050,1.1700,1.30


## Run the cleaning pipeline and inspect the report

In [6]:
cleaned, report = clean_dataset(raw)
print(report.summary())

Rows in: 6,780  ->  Rows out: 6,780
Fully duplicated rows found: 0
Duplicate Record_ID values found: 0
Nulls by column (non-zero only): {}
Negative values in non-negative fields: {}


## Findings

- 6,780 rows, 20 columns, zero missing values, zero duplicate rows or IDs.
- All categorical values (Fiscal_Quarter, Department, Expense_Category, Budget_Status) fall within their expected domains.
- No negative values in fields that should be non-negative.
- Cleaning is row-count-preserving on this dataset (6,780 in, 6,780 out) - see `docs/data_quality.md` for the full report, including the 258-record utilization outlier flag surfaced by `src/validation.py`.